In [7]:
# Import Path for handling file and folder paths in a platform-independent way.
from pathlib import Path

# Import pandas for loading, cleaning, and analyzing tabular data.
import pandas as pd

# Import NumPy for numerical operations and data manipulation.
import numpy as np

# Display all columns when viewing pandas DataFrames.
pd.set_option("display.max_columns", None)

# Display up to 100 rows when viewing pandas DataFrames.
pd.set_option("display.max_rows", 100)

# Increase the display width so wide tables are easier to read.
pd.set_option("display.width", 120)

# Display floating-point numbers with two decimal places.
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [11]:
# Get the current working directory where the notebook kernel is running.
CURRENT_DIR = Path.cwd()

# Check whether the current directory contains the datasets folder.
if (CURRENT_DIR / "datasets").exists():
    # If yes, the current directory is the project root.
    PROJECT_ROOT = CURRENT_DIR

# Otherwise, check whether the parent directory contains the datasets folder.
elif (CURRENT_DIR.parent / "datasets").exists():
    # If yes, the notebook is probably running from the notebooks folder.
    PROJECT_ROOT = CURRENT_DIR.parent

# Stop execution if the project structure cannot be detected.
else:
    raise FileNotFoundError(
        "Could not find the project root. Make sure the notebook is inside the RetailAI project."
    )

# Define the location of the raw Olist CSV files.
RAW_DATA_PATH = PROJECT_ROOT / "datasets" / "raw"

# Define the location where processed datasets will be stored.
PROCESSED_DATA_PATH = PROJECT_ROOT / "datasets" / "processed"

# Print the detected project root.
print("Project Root:", PROJECT_ROOT)

# Print the raw dataset location.
print("Raw Data Path:", RAW_DATA_PATH)

# Print the processed dataset location.
print("Processed Data Path:", PROCESSED_DATA_PATH)

Project Root: /Users/adityachaubey/Desktop/RetailAI
Raw Data Path: /Users/adityachaubey/Desktop/RetailAI/datasets/raw
Processed Data Path: /Users/adityachaubey/Desktop/RetailAI/datasets/processed


In [12]:
# Check whether the raw dataset directory exists.
if not RAW_DATA_PATH.exists():
    # Stop the notebook if the raw data directory is missing.
    raise FileNotFoundError(
        f"Raw data directory not found: {RAW_DATA_PATH}"
    )

# Find every CSV file inside the raw data directory.
csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))

# Print the total number of CSV files found.
print(f"Total CSV files found: {len(csv_files)}")

# Print the name of every discovered CSV file.
for file in csv_files:
    print("✓", file.name)

Total CSV files found: 9
✓ olist_customers_dataset.csv
✓ olist_geolocation_dataset.csv
✓ olist_order_items_dataset.csv
✓ olist_order_payments_dataset.csv
✓ olist_order_reviews_dataset.csv
✓ olist_orders_dataset.csv
✓ olist_products_dataset.csv
✓ olist_sellers_dataset.csv
✓ product_category_name_translation.csv


In [13]:
# Define the exact Olist files required for this project.
expected_files = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
}

# Convert the discovered file paths into a set of filenames.
found_files = {file.name for file in csv_files}

# Identify files that are expected but missing.
missing_files = expected_files - found_files

# Identify additional CSV files that are present but not required.
extra_files = found_files - expected_files

# Stop the notebook if any required files are missing.
if missing_files:
    raise FileNotFoundError(
        f"Missing required Olist files: {sorted(missing_files)}"
    )

# Print a success message when all required files are available.
print("✓ All required Olist datasets are available.")

# Print additional files if any exist.
if extra_files:
    print("\nAdditional CSV files found:")
    for file in sorted(extra_files):
        print("•", file)

✓ All required Olist datasets are available.


In [14]:
# Load the customer dataset.
customers = pd.read_csv(
    RAW_DATA_PATH / "olist_customers_dataset.csv"
)

# Load the Brazilian geolocation dataset.
geolocation = pd.read_csv(
    RAW_DATA_PATH / "olist_geolocation_dataset.csv"
)

# Load the order item dataset.
order_items = pd.read_csv(
    RAW_DATA_PATH / "olist_order_items_dataset.csv"
)

# Load the payment dataset.
order_payments = pd.read_csv(
    RAW_DATA_PATH / "olist_order_payments_dataset.csv"
)

# Load the customer review dataset.
order_reviews = pd.read_csv(
    RAW_DATA_PATH / "olist_order_reviews_dataset.csv"
)

# Load the order lifecycle dataset.
orders = pd.read_csv(
    RAW_DATA_PATH / "olist_orders_dataset.csv"
)

# Load the product dataset.
products = pd.read_csv(
    RAW_DATA_PATH / "olist_products_dataset.csv"
)

# Load the seller dataset.
sellers = pd.read_csv(
    RAW_DATA_PATH / "olist_sellers_dataset.csv"
)

# Load the Portuguese-to-English product category mapping.
category_translation = pd.read_csv(
    RAW_DATA_PATH / "product_category_name_translation.csv"
)

# Print confirmation that all datasets were loaded.
print("✓ All Olist datasets loaded successfully.")

✓ All Olist datasets loaded successfully.


In [15]:
# Store all datasets inside one dictionary for easier iteration and validation.
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

# Print the number of datasets loaded into memory.
print(f"Total datasets loaded: {len(datasets)}")

Total datasets loaded: 9


In [16]:
# Create a list to store summary information for every dataset.
inventory = []

# Iterate through every dataset.
for name, df in datasets.items():

    # Store basic information about the current dataset.
    inventory.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": df.memory_usage(deep=True).sum() / (1024 ** 2)
    })

# Convert the collected information into a DataFrame.
dataset_inventory = pd.DataFrame(inventory)

# Sort datasets by number of rows from largest to smallest.
dataset_inventory = dataset_inventory.sort_values(
    "rows",
    ascending=False
).reset_index(drop=True)

# Display the final dataset inventory.
dataset_inventory

,dataset,rows,columns,memory_mb
0,geolocation,1000163,5,129.38
1,order_items,112650,7,35.99
2,order_payments,103886,5,16.23
3,customers,99441,5,26.59
4,orders,99441,8,52.94
5,order_reviews,99224,7,39.12
6,products,32951,9,6.30
7,sellers,3095,4,0.59
8,category_translation,71,2,0.01


In [17]:
# Iterate through every loaded dataset.
for name, df in datasets.items():

    # Print a separator for readability.
    print("\n" + "=" * 90)

    # Print the dataset name.
    print(f"DATASET: {name.upper()}")

    # Print another separator.
    print("=" * 90)

    # Print the number of rows and columns.
    print(f"Shape: {df.shape}")

    # Print every column name.
    print("\nColumns:")

    # Display each column individually.
    for column in df.columns:
        print(f"  • {column}")


DATASET: CUSTOMERS
Shape: (99441, 5)

Columns:
  • customer_id
  • customer_unique_id
  • customer_zip_code_prefix
  • customer_city
  • customer_state

DATASET: GEOLOCATION
Shape: (1000163, 5)

Columns:
  • geolocation_zip_code_prefix
  • geolocation_lat
  • geolocation_lng
  • geolocation_city
  • geolocation_state

DATASET: ORDER_ITEMS
Shape: (112650, 7)

Columns:
  • order_id
  • order_item_id
  • product_id
  • seller_id
  • shipping_limit_date
  • price
  • freight_value

DATASET: ORDER_PAYMENTS
Shape: (103886, 5)

Columns:
  • order_id
  • payment_sequential
  • payment_type
  • payment_installments
  • payment_value

DATASET: ORDER_REVIEWS
Shape: (99224, 7)

Columns:
  • review_id
  • order_id
  • review_score
  • review_comment_title
  • review_comment_message
  • review_creation_date
  • review_answer_timestamp

DATASET: ORDERS
Shape: (99441, 8)

Columns:
  • order_id
  • customer_id
  • order_status
  • order_purchase_timestamp
  • order_approved_at
  • order_delivered_carr

In [18]:
# Iterate through every dataset.
for name, df in datasets.items():

    # Print a separator.
    print("\n" + "=" * 90)

    # Print the dataset name.
    print(f"DATA TYPES: {name.upper()}")

    # Print another separator.
    print("=" * 90)

    # Create a small table containing column names and their data types.
    dtype_table = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values
    })

    # Display the data type table.
    display(dtype_table)


DATA TYPES: CUSTOMERS


,column,dtype
0,customer_id,object
1,customer_unique_id,object
2,customer_zip_code_prefix,int64
3,customer_city,object
4,customer_state,object



DATA TYPES: GEOLOCATION


,column,dtype
0,geolocation_zip_code_prefix,int64
1,geolocation_lat,float64
2,geolocation_lng,float64
3,geolocation_city,object
4,geolocation_state,object



DATA TYPES: ORDER_ITEMS


,column,dtype
0,order_id,object
1,order_item_id,int64
2,product_id,object
3,seller_id,object
4,shipping_limit_date,object
5,price,float64
6,freight_value,float64


... (8 outputs left out)


DATA TYPES: SELLERS


,column,dtype
0,seller_id,object
1,seller_zip_code_prefix,int64
2,seller_city,object
3,seller_state,object



DATA TYPES: CATEGORY_TRANSLATION


,column,dtype
0,product_category_name,object
1,product_category_name_english,object


In [19]:
# Create an empty list to store missing-value information.
missing_reports = []

# Iterate through every dataset.
for name, df in datasets.items():

    # Calculate the number of missing values in each column.
    missing_count = df.isna().sum()

    # Calculate the percentage of missing values in each column.
    missing_percentage = (
        missing_count / len(df) * 100
    ).round(2)

    # Create a DataFrame containing missing-value statistics.
    temp = pd.DataFrame({
        "dataset": name,
        "column": missing_count.index,
        "missing_count": missing_count.values,
        "missing_percentage": missing_percentage.values
    })

    # Add the current dataset's report to the master list.
    missing_reports.append(temp)

# Combine all dataset reports into one DataFrame.
missing_report = pd.concat(
    missing_reports,
    ignore_index=True
)

# Keep only columns that contain missing values.
missing_report = missing_report[
    missing_report["missing_count"] > 0
]

# Sort columns by missing percentage.
missing_report = missing_report.sort_values(
    "missing_percentage",
    ascending=False
).reset_index(drop=True)

# Display the first 30 missing-value records.
missing_report.head(30)

,dataset,column,missing_count,missing_percentage
0,order_reviews,review_comment_title,87656,88.34
1,order_reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


In [20]:
# Create an empty list to store duplicate statistics.
duplicate_reports = []

# Iterate through every dataset.
for name, df in datasets.items():

    # Count completely duplicated rows.
    duplicate_count = df.duplicated().sum()

    # Store the result.
    duplicate_reports.append({
        "dataset": name,
        "duplicate_rows": duplicate_count
    })

# Convert the results into a DataFrame.
duplicate_report = pd.DataFrame(
    duplicate_reports
)

# Display duplicate statistics.
duplicate_report

,dataset,duplicate_rows
0,customers,0
1,geolocation,261831
2,order_items,0
3,order_payments,0
4,order_reviews,0
5,orders,0
6,products,0
7,sellers,0
8,category_translation,0


In [21]:
# Define the review columns that are important for our review intelligence system.
review_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

# Verify that all expected review columns exist.
missing_review_columns = [
    column
    for column in review_columns
    if column not in order_reviews.columns
]

# Stop execution if any required review column is missing.
if missing_review_columns:
    raise ValueError(
        f"Missing review columns: {missing_review_columns}"
    )

# Display the first 10 rows of the important review fields.
order_reviews[review_columns].head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,NaN,NaN,2018-04-13 00:00:00,2018-04-16 00:39:37
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,NaN,NaN,2017-07-16 00:00:00,2017-07-18 19:30:34
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14 00:00:00,2018-08-14 21:36:06
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,NaN,NaN,2017-05-17 00:00:00,2017-05-18 12:05:37
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47


In [22]:
# Calculate the total number of reviews.
total_reviews = len(order_reviews)

# Count reviews that contain a title.
reviews_with_title = (
    order_reviews["review_comment_title"]
    .notna()
    .sum()
)

# Count reviews that contain a text message.
reviews_with_message = (
    order_reviews["review_comment_message"]
    .notna()
    .sum()
)

# Count reviews that contain both title and message.
reviews_with_both = (
    order_reviews["review_comment_title"].notna()
    &
    order_reviews["review_comment_message"].notna()
).sum()

# Count reviews without a text message.
reviews_without_message = (
    order_reviews["review_comment_message"]
    .isna()
    .sum()
)

# Create a summary table.
review_text_summary = pd.DataFrame({
    "metric": [
        "Total reviews",
        "Reviews with title",
        "Reviews with message",
        "Reviews with both title and message",
        "Reviews without message"
    ],
    "count": [
        total_reviews,
        reviews_with_title,
        reviews_with_message,
        reviews_with_both,
        reviews_without_message
    ]
})

# Calculate the percentage of reviews containing text.
review_text_summary["percentage"] = (
    review_text_summary["count"]
    / total_reviews
    * 100
).round(2)

# Display the review text summary.
review_text_summary

,metric,count,percentage
0,Total reviews,99224,100.00
1,Reviews with title,11568,11.66
2,Reviews with message,40977,41.30
3,Reviews with both title and message,9839,9.92
4,Reviews without message,58247,58.70


In [23]:
# Select only reviews that contain an actual customer message.
review_text_data = order_reviews[
    order_reviews["review_comment_message"].notna()
].copy()

# Keep only the fields useful for initial inspection.
review_text_data = review_text_data[
    [
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
]

# Display 20 actual customer reviews.
review_text_data.head(20)

,order_id,review_score,review_comment_title,review_comment_message
3,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.
4,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...
9,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...
12,9d6f15f95d01e79bd1349cc208361f09,4,NaN,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
15,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Vendedor confiável, produto ok e entrega antes..."
16,0dacf04c5ad59fd5a0cc1faa07c34e39,2,NaN,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E..."
19,583174fbe37d3d5f0d6661be3aad1786,1,Não chegou meu produto,Péssimo
22,4fc44d78867142c627497b60a7e0228a,5,Ótimo,Loja nota 10
24,79832b7cb59ac6f887088ffd686e1d5e,5,NaN,obrigado pela atençao amim dispensada
27,2ca73e2ff9e3a186ad1e1ffb9b1d9c10,5,NaN,A compra foi realizada facilmente.\r\nA entreg...


In [24]:
# Count how many reviews exist for each review score.
review_score_distribution = (
    order_reviews["review_score"]
    .value_counts()
    .sort_index()
)

# Convert the result into a DataFrame.
review_score_distribution = (
    review_score_distribution
    .rename_axis("review_score")
    .reset_index(name="review_count")
)

# Calculate the percentage of reviews for each score.
review_score_distribution["percentage"] = (
    review_score_distribution["review_count"]
    / len(order_reviews)
    * 100
).round(2)

# Display the distribution.
review_score_distribution

,review_score,review_count,percentage
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


In [25]:
# Create a function to calculate uniqueness statistics for an ID column.
def id_quality_report(df, id_column):

    # Count total rows in the DataFrame.
    total_rows = len(df)

    # Count unique values in the ID column.
    unique_values = df[id_column].nunique()

    # Count missing ID values.
    missing_values = df[id_column].isna().sum()

    # Calculate the number of duplicated ID records.
    duplicate_values = df[id_column].duplicated().sum()

    # Return all statistics in dictionary form.
    return {
        "column": id_column,
        "rows": total_rows,
        "unique_values": unique_values,
        "missing_values": missing_values,
        "duplicate_records": duplicate_values
    }


# Check customer_id because it identifies customer records.
customer_id_quality = id_quality_report(
    customers,
    "customer_id"
)

# Check order_id because it identifies orders.
order_id_quality = id_quality_report(
    orders,
    "order_id"
)

# Check product_id because it identifies products.
product_id_quality = id_quality_report(
    products,
    "product_id"
)

# Check seller_id because it identifies sellers.
seller_id_quality = id_quality_report(
    sellers,
    "seller_id"
)

# Combine all ID quality reports.
id_quality = pd.DataFrame([
    customer_id_quality,
    order_id_quality,
    product_id_quality,
    seller_id_quality
])

# Display the results.
id_quality

,column,rows,unique_values,missing_values,duplicate_records
0,customer_id,99441,99441,0,0
1,order_id,99441,99441,0,0
2,product_id,32951,32951,0,0
3,seller_id,3095,3095,0,0


In [26]:
# Check whether every customer_id in the orders table exists in the customers table.
customer_match = orders["customer_id"].isin(
    customers["customer_id"]
)

# Calculate the percentage of orders having a valid customer reference.
customer_match_rate = customer_match.mean() * 100

# Print the relationship validation result.
print(
    f"Orders with valid customer_id: "
    f"{customer_match_rate:.2f}%"
)

Orders with valid customer_id: 100.00%


In [27]:
# Check whether every order_id in order_items exists in the orders table.
order_item_order_match = order_items["order_id"].isin(
    orders["order_id"]
)

# Calculate the valid order reference percentage.
order_item_order_rate = (
    order_item_order_match.mean() * 100
)

# Print the result.
print(
    f"Order items with valid order_id: "
    f"{order_item_order_rate:.2f}%"
)


# Check whether every order_id in reviews exists in the orders table.
review_order_match = order_reviews["order_id"].isin(
    orders["order_id"]
)

# Calculate the valid review order reference percentage.
review_order_rate = (
    review_order_match.mean() * 100
)

# Print the result.
print(
    f"Reviews with valid order_id: "
    f"{review_order_rate:.2f}%"
)

Order items with valid order_id: 100.00%
Reviews with valid order_id: 100.00%


In [28]:
# Check whether every product_id in order_items exists in products.
product_match = order_items["product_id"].isin(
    products["product_id"]
)

# Calculate the valid product reference percentage.
product_match_rate = product_match.mean() * 100

# Print the result.
print(
    f"Order items with valid product_id: "
    f"{product_match_rate:.2f}%"
)


# Check whether every seller_id in order_items exists in sellers.
seller_match = order_items["seller_id"].isin(
    sellers["seller_id"]
)

# Calculate the valid seller reference percentage.
seller_match_rate = seller_match.mean() * 100

# Print the result.
print(
    f"Order items with valid seller_id: "
    f"{seller_match_rate:.2f}%"
)

Order items with valid product_id: 100.00%
Order items with valid seller_id: 100.00%


In [29]:
# Define date columns available in the orders dataset.
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert order date columns from strings to pandas datetime objects.
for column in order_date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )


# Define date columns available in the reviews dataset.
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

# Convert review date columns to datetime.
for column in review_date_columns:
    order_reviews[column] = pd.to_datetime(
        order_reviews[column],
        errors="coerce"
    )


# Convert shipping limit date into datetime.
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

# Print confirmation.
print("✓ Date columns converted to datetime.")

✓ Date columns converted to datetime.


In [30]:
# Display the data types of the converted order date columns.
orders[order_date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [31]:
# Create a mask for orders where delivery happened before purchase.
invalid_delivery_sequence = (
    orders["order_delivered_customer_date"]
    <
    orders["order_purchase_timestamp"]
)

# Count invalid timeline records.
invalid_delivery_count = invalid_delivery_sequence.sum()

# Print the number of invalid records.
print(
    f"Orders with delivery before purchase: "
    f"{invalid_delivery_count}"
)

Orders with delivery before purchase: 0


In [32]:
# Create a summary of the major data quality checks.
data_quality_summary = pd.DataFrame({
    "check": [
        "Required CSV files",
        "Customers loaded",
        "Orders loaded",
        "Products loaded",
        "Sellers loaded",
        "Reviews loaded",
        "Review messages available",
        "Invalid delivery sequences"
    ],

    "result": [
        len(csv_files) == len(expected_files),
        len(customers),
        len(orders),
        len(products),
        len(sellers),
        len(order_reviews),
        reviews_with_message,
        invalid_delivery_count
    ]
})

# Display the final quality summary.
data_quality_summary

,check,result
0,Required CSV files,True
1,Customers loaded,99441
2,Orders loaded,99441
3,Products loaded,32951
4,Sellers loaded,3095
5,Reviews loaded,99224
6,Review messages available,40977
7,Invalid delivery sequences,0


In [33]:
# Create the processed-data directory if it does not already exist.
PROCESSED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Print confirmation.
print(
    f"✓ Processed data directory ready: "
    f"{PROCESSED_DATA_PATH}"
)

✓ Processed data directory ready: /Users/adityachaubey/Desktop/RetailAI/datasets/processed


In [34]:
# Create a dictionary containing datasets that have been processed in this notebook.
processed_datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

# Save every processed dataset as a Parquet file.
for name, df in processed_datasets.items():

    # Create the output file path.
    output_path = PROCESSED_DATA_PATH / f"{name}.parquet"

    # Save the DataFrame without storing the pandas index.
    df.to_parquet(
        output_path,
        index=False
    )

    # Print the saved file path.
    print(f"✓ Saved: {output_path}")

✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/customers.parquet
✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/geolocation.parquet
✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/order_items.parquet
✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/order_payments.parquet
✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/order_reviews.parquet
✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/orders.parquet
✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/products.parquet
✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/sellers.parquet
✓ Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/category_translation.parquet
